<div style="text-align: center;">
  <h1 style="text-align: center;">Workshop: Prototipado de Agentes con OpenAI Agents SDK</h1>
  <h3 style="text-align: center;">Máster en Data Science e IA · EBIS Business Techschool</h3>
  <p style="text-align: center;">Bloque A — Fundamentos del SDK</p>
</div>

## Objetivos de aprendizaje

En este notebook aprenderás:
- Qué son los agentes de IA y por qué importan
- Los 4 conceptos fundamentales del OpenAI Agents SDK: Agents, Handoffs, Guardrails, Sessions
- Cómo construir agentes con tools, validación y memoria — sobre el caso de una agencia de viajes inteligente

## Caso transversal

Construiremos **TravelMind**, un asistente de viajes que ayuda a planificar escapadas: calcula presupuestos, consulta clima y vuelos, recomienda destinos y en el Bloque B lo conectaremos con guías de viaje reales.

### 🧭 Antes de empezar

- **Cómo se usa:** ejecuta las celdas en orden, de arriba abajo, con `Mayús + Enter`. Si una celda da error, lee la última línea: casi siempre dice qué falta.
- **Si algo se queda raro:** menú *Entorno de ejecución → Reiniciar y ejecutar todo*.
- **Tu API key:** en Colab, guárdala como secreto `OPENAI_API_KEY` (icono de la llave, a la izquierda). Nunca la pegues dentro de una celda.
- **`async`, `await` y `nest_asyncio`:** aparecen porque el SDK es asíncrono. No hace falta entenderlos para seguir el lab.

**Vocabulario de este notebook**

| Término | Qué es |
|---|---|
| **`Agent` y `Runner`** | el agente (instrucciones, modelo y herramientas) y quien lo ejecuta. |
| **`@function_tool`** | convierte una función de Python en una herramienta que el agente puede usar. |
| **`output_type`** | obliga al agente a responder con un modelo de Pydantic (salida estructurada). |
| **Handoff** | un agente pasa la conversación a otro más especializado. |
| **Guardrail** | un filtro antes o después del agente, como «solo preguntas de viajes». |
| **Session** | la memoria de la conversación entre llamadas. |


## Configuración

Instala las dependencias y configura el entorno. **Después de instalar, reinicia el kernel** antes de continuar.

In [ ]:
!pip install -q --upgrade "openai-agents>=0.24,<0.25" nest_asyncio pandas

In [ ]:
import os
from getpass import getpass

# La clave NUNCA se escribe en el notebook: Colab (secretos) -> variable de entorno -> la pides por teclado
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

os.environ["OPENAI_DEFAULT_MODEL"] = "gpt-6-luna"   # modelo por defecto de todos los agentes (el GPT-6 más barato)


In [ ]:
import nest_asyncio
nest_asyncio.apply()
print("✅ Listo.")

## 1. ¿Qué son los Agentes de IA?

Un **chatbot** responde con lo que sabe. Un **agente** puede ir a buscar la información, calcularla, o actuar sobre un sistema externo — y luego responder.

La diferencia no es el modelo de lenguaje — es la **arquitectura**:

```
── CHATBOT ──────────────────────────────────────────────────
  Usuario  →  "¿Cuánto cuesta volar a Tokio?"
  Modelo   →  "Suele rondar los 600-900€ según fecha y aerolínea."
               ↑ estimación razonable, pero inventada —
                 el modelo no tiene acceso a precios reales

── AGENTE ───────────────────────────────────────────────────
  Usuario  →  "¿Cuánto cuesta volar a Tokio?"
  Agente   →  (decide que necesita datos reales)
           →  buscar_vuelos("Madrid", "Tokio")  ← llama al código
  Código   →  {"precio": 780, "aerolinea": "Iberia/JAL"}
  Agente   →  "El vuelo más barato son 780€ con Iberia/JAL."
               ↑ misma pregunta, mismo modelo,
                 pero la respuesta viene de datos reales
```

El modelo de lenguaje es el mismo en los dos casos. Lo que cambia es que el agente tiene **herramientas** (funciones Python) y un **bucle** que le permite usarlas antes de responder.

### Las 4 capacidades que definen a un agente

| Capacidad | Qué significa en la práctica |
|---|---|
| **Razonar** | Decide qué tool usar y en qué orden. "Para comparar Lisboa y Londres necesito clima Y vuelos." |
| **Actuar** | Llama a funciones Python reales y procesa sus resultados. |
| **Decidir** | Si un vuelo no existe, lo dice. No inventa datos. |
| **Recordar** | Sabe que "esa ciudad" es Tokio porque lo dijiste dos turnos antes. |

### Por qué los agentes importan más allá del chat

Un LLM tiene conocimiento estático hasta su fecha de corte y no sabe nada de tu empresa. Un agente puede:

- **Consultar datos en vivo**: precios de vuelos, disponibilidad de hoteles, tipo de cambio del día
- **Operar sobre sistemas reales**: hacer una reserva, actualizar una base de datos, enviar un email
- **Razonar sobre información privada**: documentos internos, historial del usuario, datos de empresa que el modelo nunca ha visto

En el Reto del Bloque B construirás un agente que responde preguntas sobre guías de viaje internas — algo imposible para un LLM genérico sin acceso a esos PDFs.

## 2. OpenAI Agents SDK (versión 0.24, noviembre 2026)

El SDK oficial de OpenAI se apoya en 4 primitivas (las versiones recientes añaden además *Sandbox Agents* para ejecutar código en un entorno aislado). Todo lo que veremos hoy se construye con estas 4 piezas:

| Primitiva | Para qué sirve | Ejemplo |
|---|---|---|
| **Agent** | Define quién es el agente: nombre, instrucciones, tools | `agente_presupuesto = Agent(tools=[calcular_presupuesto])` |
| **Handoff** | Pasa el control a otro agente más especializado | El coordinador delega al especialista en visados |
| **Guardrail** | Filtra entradas o valida salidas | Rechazar preguntas off-topic antes de gastar tokens |
| **Session** | Mantiene el historial entre llamadas | El agente recuerda que viajáis con una niña de 8 años |

### ¿Por qué un SDK y no la API directa?

Con la API directa tendrías que escribir tú el bucle del agente (es lo que hicimos a mano en el reto del agente SQL):

```python
# Con la API directa — ~40 líneas para un agente simple con tools
r = client.responses.create(model="gpt-6-luna", input=pregunta, tools=tools_json)
while llamadas := [o for o in r.output if o.type == "function_call"]:
    salidas = [{"type": "function_call_output", "call_id": ll.call_id,
                "output": ejecutar_tool(ll.name, ll.arguments)} for ll in llamadas]
    r = client.responses.create(model="gpt-6-luna", previous_response_id=r.id, input=salidas, tools=tools_json)
print(r.output_text)  # respuesta final

# Con el Agents SDK — mismo resultado en ~5 líneas
agente = Agent(name="...", instructions="...", tools=[mi_tool])
resultado = Runner.run_sync(agente, "¿Cuánto cuesta volar a Tokio?")
print(resultado.final_output)
```

El SDK elimina el boilerplate y expone solo lo que importa: qué hace el agente, no cómo se orquesta.

## 3. Cómo funciona un agente por dentro

Cuando llamas a `Runner.run_sync()` arranca un **bucle de razonamiento**. El modelo decide en cada paso si necesita más información (tool call) o si ya puede responder:

```
Tu pregunta: "Tengo 1200€ para 7 días en Tailandia, ¿cómo distribuyo el presupuesto?"
    │
    ▼
┌─────────────────────────────────────────────────────────┐
│  Runner.run_sync()                                       │
│                                                          │
│  Turno 1:                                                │
│    Modelo recibe la pregunta                             │
│    Decide: "Necesito calcular el presupuesto"            │
│    → llama a calcular_presupuesto_diario(1200, 7)        │
│    ← recibe: {diario: 171.4, alojamiento: 68.6, ...}    │
│                                                          │
│  Turno 2:                                                │
│    Modelo recibe la pregunta + el resultado de la tool   │
│    Decide: "Ya tengo suficiente información"             │
│    → genera respuesta final en lenguaje natural          │
└─────────────────────────────────────────────────────────┘
    │
    ▼
resultado.final_output  →  "Con 1200€ para 7 días tendrás 68€/día para alojamiento..."
```

**Puntos clave:**
- El modelo **decide solo** qué tools usar — tú no lo programas explícitamente
- El bucle continúa hasta que el modelo considera que tiene suficiente información
- Si una tool falla o devuelve un error, el modelo lo procesa y puede intentar otra estrategia
- `Runner.run_sync()` bloquea hasta que el bucle termina; `Runner.run()` es la versión async

## 4. Primer Agente: sin tools

El agente más simple: solo `name` + `instructions`. Sin tools, el modelo usa su conocimiento interno.

Fíjate en el `instructions` — es el **system prompt**. Define la personalidad, el dominio y las restricciones del agente. Es el único lugar donde hablas directamente con el modelo antes de que el usuario intervenga.

In [ ]:
from agents import Agent, Runner

agente_simple = Agent(
    name="TravelMind Basic",
    instructions=(
        "Eres TravelMind, un asistente experto en viajes. "
        "Ayudas a planificar viajes, recomiendas destinos y explicas qué esperar de cada lugar. "
        "Sé concreto, práctico y entusiasta. Si no estás seguro, dilo."
    )
)

resultado = Runner.run_sync(
    agente_simple,
    "¿Qué diferencia hay entre visitar Japón en marzo vs en octubre?"
)

print(resultado.final_output)

`resultado.final_output` es siempre el punto de entrada al resultado.

- Sin `output_type` → es un `str` (texto libre como el de arriba)
- Con `output_type=MiClase` → es una instancia de esa clase (lo veremos en la sección 7)

El agente anterior no tiene tools, así que responde con su conocimiento de entrenamiento. Útil para preguntas generales, pero no puede consultar datos en tiempo real ni acceder a información privada.

## 5. Agentes con Tools

Una **tool** es una función Python normal decorada con `@function_tool`. El modelo lee el docstring para entender qué hace y cuándo usarla — no hay que decirle explícitamente "usa esta tool ahora".

**Cómo el modelo sabe qué tool usar:**
```
Pregunta: "Tengo 1200€ para 7 días en Tailandia..."

El modelo lee las tools disponibles:
  - calcular_presupuesto_diario: "Distribuye un presupuesto de viaje..."  ← esto encaja
  - convertir_divisa: "Convierte un importe entre divisas..."
  - dias_hasta_viaje: "Calcula cuántos días quedan..."

Decide llamar a calcular_presupuesto_diario(presupuesto_total=1200, num_dias=7)
```

El **docstring** es lo que el modelo lee. Los **type hints** generan el schema JSON automáticamente — el modelo sabe exactamente qué parámetros pasar y de qué tipo.

In [ ]:
from datetime import date
from agents import function_tool

@function_tool
def calcular_presupuesto_diario(
    presupuesto_total: float,
    num_dias: int,
    porcentaje_alojamiento: float = 40.0
) -> dict:
    """Distribuye un presupuesto de viaje entre alojamiento, comida y actividades.

    Args:
        presupuesto_total: Presupuesto total en euros.
        num_dias: Duración del viaje en días.
        porcentaje_alojamiento: Porcentaje destinado a alojamiento (por defecto 40%).
    """
    print("  → calcular_presupuesto_diario llamada")
    diario = presupuesto_total / num_dias
    alojamiento = diario * porcentaje_alojamiento / 100
    comida = diario * 0.35
    actividades = diario - alojamiento - comida
    return {
        "diario_total": round(diario, 2),
        "alojamiento": round(alojamiento, 2),
        "comida": round(comida, 2),
        "actividades": round(actividades, 2),
    }

@function_tool
def convertir_divisa(importe: float, moneda_origen: str, moneda_destino: str) -> float:
    """Convierte un importe entre divisas usando tasas de referencia aproximadas.

    Args:
        importe: Cantidad a convertir.
        moneda_origen: Código ISO de la moneda origen (EUR, USD, JPY, GBP, THB, MXN).
        moneda_destino: Código ISO de la moneda destino.
    """
    print(f"  → convertir_divisa {moneda_origen} → {moneda_destino}")
    tasas = {"EUR": 1.0, "USD": 1.08, "JPY": 160.0, "GBP": 0.86, "THB": 38.5, "MXN": 18.5}
    if moneda_origen not in tasas or moneda_destino not in tasas:
        return -1.0
    en_eur = importe / tasas[moneda_origen]
    return round(en_eur * tasas[moneda_destino], 2)

@function_tool
def dias_hasta_viaje(fecha_salida: str) -> int:
    """Calcula cuántos días quedan hasta la fecha de salida.

    Args:
        fecha_salida: Fecha en formato YYYY-MM-DD.
    """
    print(f"  → dias_hasta_viaje {fecha_salida}")
    salida = date.fromisoformat(fecha_salida)
    return (salida - date.today()).days

agente_presupuesto = Agent(
    name="TravelMind Presupuesto",
    instructions=(
        "Eres un planificador de viajes especializado en presupuestos. "
        "Usa las herramientas para calcular distribuciones de gasto, convertir divisas "
        "y calcular cuánto tiempo queda para preparar el viaje. Sé concreto con los números."
    ),
    tools=[calcular_presupuesto_diario, convertir_divisa, dias_hasta_viaje]
)

resultado = Runner.run_sync(
    agente_presupuesto,
    "Tengo 1200€ para 7 días en Tailandia. ¿Cómo distribuyo el presupuesto y a cuántos baht equivale lo de alojamiento?"
)

print(resultado.final_output)

Fíjate en los prints `→ tool llamada` — puedes ver exactamente qué tools eligió el agente y en qué orden. Ahora probamos con una pregunta que requiere **varias tools a la vez**:

In [ ]:
resultado = Runner.run_sync(
    agente_presupuesto,
    "Me voy a Japón el 2026-09-15 con 2000€ para 10 días. "
    "¿Cuánto me queda de presupuesto diario para actividades y a cuántos yenes equivale?"
)

print(resultado.final_output)

## 6. Tools async: llamadas en paralelo

Las tools síncronas se ejecutan de una en una. Las tools `async` permiten al SDK **lanzarlas en paralelo** cuando el modelo decide que las necesita todas.

```
Sin async (secuencial):           Con async (paralelo):
  t=0.0s  consultar_clima(Lisboa)   t=0.0s  consultar_clima(Lisboa)    ┐
  t=0.8s  consultar_vuelos(Lisboa)          consultar_vuelos(Lisboa)    │ simultáneas
  t=1.6s  consultar_clima(Londres)          consultar_clima(Londres)    │
  t=2.4s  consultar_vuelos(Londres)         consultar_vuelos(Londres)   ┘
  t=3.2s  respuesta final           t=0.8s  respuesta final
```

La mejora es enorme cuando las tools hacen llamadas a APIs externas (bases de datos, servicios web). En local con datos mock no se nota tanto, pero en producción puede pasar de 10s a 2s.

> `async` no significa "más rápido" en CPU — significa "no bloqueo mientras espero". El SDK aprovecha esas esperas para lanzar otras tools.

In [ ]:
import asyncio
import time

@function_tool
async def consultar_clima(ciudad: str) -> dict:
    """Consulta el clima actual y la probabilidad de lluvia de una ciudad.

    Args:
        ciudad: Nombre de la ciudad.
    """
    await asyncio.sleep(0.8)  # simula 0.8s de latencia de una API real
    print(f"  → consultar_clima({ciudad}) terminó")
    datos = {
        "Lisboa": {"temp_c": 22, "condicion": "soleado", "lluvia_prob_pct": 10},
        "Londres": {"temp_c": 14, "condicion": "nublado", "lluvia_prob_pct": 65},
        "Bangkok": {"temp_c": 34, "condicion": "húmedo", "lluvia_prob_pct": 40},
        "Tokio":   {"temp_c": 18, "condicion": "variable", "lluvia_prob_pct": 25},
    }
    return datos.get(ciudad, {"temp_c": 20, "condicion": "variable", "lluvia_prob_pct": 30})

@function_tool
async def consultar_vuelos(origen: str, destino: str) -> dict:
    """Consulta disponibilidad y precio mínimo de vuelos directos entre dos ciudades.

    Args:
        origen: Ciudad de origen.
        destino: Ciudad de destino.
    """
    await asyncio.sleep(0.8)  # simula 0.8s de latencia
    print(f"  → consultar_vuelos({origen}→{destino}) terminó")
    rutas = {
        ("Madrid", "Lisboa"):  {"directo": True,  "precio_min_eur": 59,  "aerolineas": ["Iberia", "Vueling"]},
        ("Madrid", "Londres"): {"directo": True,  "precio_min_eur": 89,  "aerolineas": ["Iberia", "British Airways"]},
        ("Madrid", "Bangkok"): {"directo": False, "precio_min_eur": 520, "aerolineas": ["Emirates", "Qatar"]},
        ("Madrid", "Tokio"):   {"directo": False, "precio_min_eur": 680, "aerolineas": ["JAL", "ANA"]},
    }
    return rutas.get((origen, destino), {"directo": False, "precio_min_eur": 400, "aerolineas": ["varias"]})

agente_comparador = Agent(
    name="TravelMind Comparador",
    instructions=(
        "Cuando compares destinos, consulta clima y vuelos en paralelo para ahorrar tiempo. "
        "Resume la comparativa de forma clara y recomienda una opción."
    ),
    tools=[consultar_clima, consultar_vuelos],
)

t0 = time.perf_counter()
resultado = Runner.run_sync(
    agente_comparador,
    "Quiero irme este finde desde Madrid. ¿Lisboa o Londres? Dime clima y precio de vuelo de cada una."
)
print(resultado.final_output)
print(f"\n⏱  Total: {time.perf_counter() - t0:.2f}s")
print("   (Si fueran secuenciales: 0.8 + 0.8 + 0.8 + 0.8 = 3.2s. En paralelo: ~0.8s)")

## 7. Salida estructurada con Pydantic

Por defecto `final_output` es texto libre (`str`). Pero en producción casi siempre necesitas **parsear** esa respuesta para guardarla, mostrarla en una UI, o pasarla a otro sistema.

Con `output_type=MiClase` el modelo rellena un formulario estructurado — no hace falta parsear nada:

```python
# Sin output_type — tienes que parsear texto a mano:
r = resultado.final_output  # "Te recomiendo Roma en octubre, es cultural y de precio medio..."
destino = r.split("recomiendo ")[1].split(" en")[0]  # frágil y feo

# Con output_type — acceso directo por atributo:
r = resultado.final_output  # RecomendacionViaje(destino="Roma", tipo="cultural", ...)
print(r.destino)   # "Roma"
print(r.tipo)      # "cultural"
```

Pydantic también **valida** los valores: si defines `tipo: Literal["playa", "ciudad", ...]` el modelo nunca podrá devolver un valor que no esté en esa lista. Si intenta hacerlo, el SDK lanza un error antes de devolverte el resultado.

In [ ]:
from typing import Literal
from pydantic import BaseModel

class RecomendacionViaje(BaseModel):
    destino: str
    tipo: Literal["playa", "ciudad", "naturaleza", "aventura", "cultural"]
    presupuesto_nivel: Literal["bajo", "medio", "alto"]
    mejor_epoca: str
    razon: str

agente_recomendador = Agent(
    name="TravelMind Recomendador",
    instructions=(
        "Recomienda el destino más adecuado para el perfil del viajero. "
        "Sé específico con el destino (ciudad o región, no solo país)."
    ),
    output_type=RecomendacionViaje,
)

resultado = Runner.run_sync(
    agente_recomendador,
    "Somos una pareja, nos encanta la historia y la gastronomía, presupuesto medio, 10 días en octubre."
)

rec = resultado.final_output  # ya no es str — es un objeto RecomendacionViaje
print(f"type(final_output) = {type(rec).__name__}")
print()
print(f"✈️  Destino:       {rec.destino}")
print(f"🏷️  Tipo:          {rec.tipo}")
print(f"💶  Presupuesto:   {rec.presupuesto_nivel}")
print(f"📅  Mejor época:   {rec.mejor_epoca}")
print(f"💬  Razón:         {rec.razon}")

## 8. Handoffs: delegar a un especialista

Un único agente con instrucciones muy largas tiende a perder calidad — el modelo mezcla dominios y se vuelve genérico. La solución es dividir: un **coordinador** que enruta y **especialistas** que responden.

```
Sin handoffs — un agente que lo sabe todo (peor calidad):
  Coordinador → "¿Qué hotel en Trastevere o centro de Roma?" → respuesta genérica

Con handoffs — el coordinador enruta al experto correcto:
  Coordinador → detecta "alojamiento" → transfiere al Especialista en Alojamiento
  Especialista → responde con profundidad sobre zonas, precios, relación calidad-precio
```

**Por qué mejora la calidad:** cada especialista tiene un system prompt enfocado en su dominio, sin ruido de otros temas. El modelo puede ser mucho más preciso cuando solo tiene que pensar en una cosa.

**Cómo funciona técnicamente:** `handoff(agent=otro_agente)` crea una tool especial. Cuando el coordinador la llama, el Runner pasa el historial completo al especialista y este continúa la conversación desde ahí.

In [ ]:
from agents import handoff

agente_vuelos = Agent(
    name="Especialista en Vuelos",
    instructions=(
        "Eres experto en vuelos: rutas, aerolíneas de bajo coste, escalas, equipaje, "
        "check-in online y consejos para encontrar los mejores precios."
    )
)

agente_alojamiento = Agent(
    name="Especialista en Alojamiento",
    instructions=(
        "Eres experto en alojamiento: hoteles, hostels, Airbnb, apartamentos turísticos. "
        "Conoces las mejores zonas de cada ciudad y la relación calidad-precio."
    )
)

agente_actividades = Agent(
    name="Especialista en Actividades",
    instructions=(
        "Eres experto en experiencias de viaje: tours, museos, gastronomía local, "
        "actividades para familias, rutas de senderismo e itinerarios día a día."
    )
)

agente_coordinador = Agent(
    name="TravelMind Coordinador",
    instructions=(
        "Eres el coordinador principal de TravelMind. "
        "Analiza la consulta del viajero y delega al especialista apropiado. "
        "Si la consulta cruza dominios, delega al más relevante primero."
    ),
    handoffs=[
        handoff(
            agent=agente_vuelos,
            tool_description_override="Delega para preguntas sobre aerolíneas, rutas, equipaje o precios de vuelo"
        ),
        handoff(
            agent=agente_alojamiento,
            tool_description_override="Delega para preguntas sobre hoteles, Airbnb o dónde alojarse"
        ),
        handoff(
            agent=agente_actividades,
            tool_description_override="Delega para itinerarios, qué ver, tours o gastronomía"
        ),
    ]
)

resultado = Runner.run_sync(
    agente_coordinador,
    "¿Merece la pena pagar extra por un hotel en el centro de Roma o mejor en zona Trastevere?"
)

print(resultado.final_output)

**Ventaja clave**: cada especialista tiene un system prompt ajustado a su dominio, lo que mejora la calidad de la respuesta. Añadir un nuevo experto (visados, seguros de viaje, transporte local...) es tan simple como crear un nuevo `Agent` y añadirlo a la lista de `handoffs`.

## 9. Guardrails: filtros antes y después

Los guardrails son funciones async que se ejecutan **en paralelo** con el agente (input) o **justo antes** de devolver el resultado (output).

```
Input guardrail:                    Output guardrail:
  Usuario → [guardrail] → Agente     Agente → [guardrail] → Usuario
               ↓ si falla               ↓ si falla
           Se corta la ejecución    Error antes de devolver
                                    (el agente ya corrió, pero se descarta)
```

**Casos de uso habituales:**
- **Input**: rechazar preguntas off-topic, detectar contenido inapropiado, validar formato de entrada
- **Output**: verificar que la respuesta incluye citas, que no supera X caracteres, que los datos son coherentes

> Por defecto, el guardrail de entrada corre **a la vez** que el agente: si salta, se corta, pero el agente puede haber gastado ya algunos tokens. Con `@input_guardrail(run_in_parallel=False)` el agente espera a que el filtro dé el visto bueno. Un filtro de palabras clave, como el de abajo, no gasta tokens; uno que usa un LLM, sí.

Un guardrail **no lanza excepciones**: devuelve `GuardrailFunctionOutput(output_info, tripwire_triggered)`. `False` = todo OK, deja pasar; `True` = **salta el cable trampa** y el SDK corta la ejecución lanzando `InputGuardrailTripwireTriggered` u `OutputGuardrailTripwireTriggered`, que tú capturas para responder al usuario.

En el Reto usaréis un output guardrail para exigir que cada respuesta venga acompañada de al menos una cita real de una guía.

In [ ]:
from agents import (
    input_guardrail, output_guardrail,
    GuardrailFunctionOutput, RunContextWrapper,
    InputGuardrailTripwireTriggered, OutputGuardrailTripwireTriggered,
)
from pydantic import BaseModel

class RespuestaViaje(BaseModel):
    respuesta: str
    nivel_urgencia: int  # 1 = informativa · 2 = acción recomendada · 3 = reservar ya

### Paso 2 — Definir los guardrails

In [ ]:
@input_guardrail(run_in_parallel=False)   # el agente espera al filtro
async def validar_consulta_viajes(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input_text: str,
) -> GuardrailFunctionOutput:
    """Rechaza consultas inapropiadas antes de que lleguen al agente."""
    patrones_prohibidos = [
        "hackear", "ilegal", "sin visado falso",
        "pasar la frontera sin papeles", "traficar", "contrabando",
    ]
    if any(p in input_text.lower() for p in patrones_prohibidos):
        return GuardrailFunctionOutput(output_info="consulta prohibida", tripwire_triggered=True)   # corta la ejecución
    return GuardrailFunctionOutput(output_info="OK", tripwire_triggered=False)


@output_guardrail
async def validar_longitud_movil(
    ctx: RunContextWrapper[None],
    agent: Agent,
    output: RespuestaViaje,
) -> GuardrailFunctionOutput:
    """La respuesta no puede superar 1200 caracteres (legible en móvil)."""
    longitud = len(output.respuesta)
    return GuardrailFunctionOutput(output_info=f"{longitud} caracteres", tripwire_triggered=longitud > 1200)

### Paso 3 — Crear el agente y probar

In [ ]:
agente_validado = Agent(
    name="TravelMind Validado",
    instructions=(
        "Eres TravelMind, asistente de viajes. "
        "Responde de forma breve y concreta (máximo 3-4 frases). "
        "Indica el nivel de urgencia: 1 informativa, 2 acción recomendada, 3 reservar ya."
    ),
    input_guardrails=[validar_consulta_viajes],
    output_guardrails=[validar_longitud_movil],
    output_type=RespuestaViaje,
)

def probar_consulta(titulo: str, consulta: str):
    print(f"--- {titulo} ---")
    try:
        r = Runner.run_sync(agente_validado, consulta).final_output
        print(f"✅ {r.respuesta}")
        print(f"   Urgencia: {r.nivel_urgencia}")
    except InputGuardrailTripwireTriggered as e:
        print(f"❌ Bloqueado a la entrada: {e.guardrail_result.output.output_info}")
    except OutputGuardrailTripwireTriggered as e:
        print(f"❌ Bloqueado a la salida: {e.guardrail_result.output.output_info}")
    print()

probar_consulta("Consulta válida", "¿Cuándo es mejor visitar Bangkok para evitar el monzón?")
probar_consulta("Consulta inválida", "¿Cómo puedo entrar a EEUU sin visado falso?")

## 10. Sessions: memoria entre llamadas

Sin session, cada `Runner.run_sync()` es una conversación nueva — el agente no sabe nada de lo que dijiste antes. Con `SQLiteSession`, el historial persiste entre llamadas dentro de la misma ejecución. Para que sobreviva a un reinicio del kernel hay que darle un fichero: `SQLiteSession("mi_viaje", "conversaciones.db")`; sin fichero vive en memoria y se pierde.

```python
# Sin session — el agente no recuerda:
Runner.run_sync(agente, "Planifico 14 días en Japón con una niña de 8 años.")
Runner.run_sync(agente, "¿Qué actividades en Tokio?")
# ← El agente responde actividades genéricas, sin saber que hay una niña de 8 años

# Con session — el agente recuerda el hilo:
sesion = SQLiteSession("mi_viaje_japon")
Runner.run_sync(agente, "Planifico 14 días en Japón con una niña de 8 años.", session=sesion)
Runner.run_sync(agente, "¿Qué actividades en Tokio?", session=sesion)
# ← El agente sugiere Disney Sea, Pokémon Center, kaiten-sushi... porque sabe que hay una niña
```

El `id` de la sesión (`"mi_viaje_japon"`) es la clave que identifica el hilo. En producción lo vincularías al `user_id` de tu sistema.

In [ ]:
from agents import SQLiteSession

agente_planificador = Agent(
    name="TravelMind Planificador",
    instructions=(
        "Eres el planificador personal de TravelMind. "
        "Recuerdas los detalles del viaje que el usuario va compartiendo contigo. "
        "Sé conciso y adapta siempre tu respuesta al contexto acumulado."
    )
)

sesion = SQLiteSession("viaje_japon_2026_ebis", "conversaciones.db")  # con fichero: sobrevive a reinicios

# Turno 1 — el agente aprende el contexto del viaje
r1 = Runner.run_sync(
    agente_planificador,
    "Planifico 14 días en Japón en marzo de 2026. Somos 2 adultos y 1 niña de 8 años, presupuesto total 4000€.",
    session=sesion,
)
print("Turno 1:", r1.final_output)

# Turno 2 — el agente recuerda que hay una niña de 8 años → adapta la respuesta
r2 = Runner.run_sync(
    agente_planificador,
    "¿Qué actividades recomiendas en Tokio para el primer día?",
    session=sesion,
)
print("\nTurno 2:", r2.final_output)

## 11. Ejemplo Completo: Analizador de Escapadas

Combinamos Agent + Tools + salida estructurada en un ejemplo realista: el agente analiza un dataset de destinos y recomienda el mejor según el presupuesto.

In [ ]:
import pandas as pd

opciones = pd.DataFrame({
    "destino":            ["Lisboa", "Ámsterdam", "Praga", "Dubrovnik", "Edimburgo"],
    "precio_vuelo_eur":   [89,        145,          112,     210,          130],
    "precio_hotel_noche": [95,        160,           75,     180,          110],
    "temp_media_mayo_c":  [21,         16,           19,      24,           13],
    "horas_vuelo":        [1.5,         3.0,          3.5,     2.5,          2.5],
    "valoracion_media":   [4.6,         4.4,          4.7,     4.5,          4.5],
})

@function_tool
def listar_destinos() -> list[dict]:
    """Devuelve todas las opciones de destino disponibles con precios, clima y valoraciones."""
    return opciones.to_dict(orient="records")

@function_tool
def destino_mas_barato() -> str:
    """Identifica el destino con menor coste total (vuelo ida+vuelta + 2 noches de hotel)."""
    df = opciones.copy()
    df["coste_total"] = df["precio_vuelo_eur"] * 2 + df["precio_hotel_noche"] * 2
    d = df.loc[df["coste_total"].idxmin()]
    return f"{d['destino']} con {d['coste_total']:.0f}€ totales"

@function_tool
def mejor_relacion_calidad_precio() -> dict:
    """Calcula la mejor opción ponderando valoración, precio y temperatura."""
    df = opciones.copy()
    df["score"] = (
        df["valoracion_media"] * 20
        - df["precio_vuelo_eur"] / 10
        - df["precio_hotel_noche"] / 20
        + df["temp_media_mayo_c"] / 2
    )
    d = df.loc[df["score"].idxmax()]
    return {"destino": d["destino"], "score": round(d["score"], 1), "temp_c": d["temp_media_mayo_c"]}

@function_tool
def filtrar_por_presupuesto(presupuesto_max_eur: float) -> list[dict]:
    """Filtra destinos que caben en el presupuesto máximo (vuelo ida+vuelta + 2 noches).

    Args:
        presupuesto_max_eur: Presupuesto máximo total en euros.
    """
    df = opciones.copy()
    df["coste_total"] = df["precio_vuelo_eur"] * 2 + df["precio_hotel_noche"] * 2
    validos = df[df["coste_total"] <= presupuesto_max_eur]
    return validos[["destino", "coste_total", "temp_media_mayo_c", "valoracion_media"]].to_dict(orient="records")

agente_analizador = Agent(
    name="TravelMind Analizador",
    instructions=(
        "Eres el analizador de escapadas de fin de semana de TravelMind. "
        "Usa las tools para comparar destinos y da una recomendación clara, justificada con datos."
    ),
    tools=[listar_destinos, destino_mas_barato, mejor_relacion_calidad_precio, filtrar_por_presupuesto],
)

resultado = Runner.run_sync(
    agente_analizador,
    "Tengo un finde libre en mayo, salgo desde Madrid con 400€ de presupuesto máximo. ¿A dónde me voy?"
)

print(resultado.final_output)

## 12. Más allá de los fundamentos

Cuatro características del SDK útiles en producción. Las vemos brevemente — en el Bloque B usaremos algunas de ellas.

### 12.1 Tracing automático

Cada `Runner.run_sync()` genera automáticamente una traza en `platform.openai.com/traces`: qué tools se llamaron, qué respondió el modelo en cada paso, cuánto tardó. Sin código extra — solo tienes que mirar el dashboard.

In [ ]:
import os
from agents import trace

with trace("demo workshop travelmind") as t:
    res = Runner.run_sync(
        agente_simple,
        "¿Cuáles son los 3 destinos más populares de Europa para un viaje cultural?"
    )

print(res.final_output)
print(f"\n🔗 https://platform.openai.com/traces/trace?trace_id={t.trace_id}")

### 12.2 Streaming

`Runner.run_streamed()` muestra la respuesta **token a token** mientras se genera, en lugar de esperar a que termine. Ideal para interfaces de chat.

In [ ]:
import asyncio
from openai.types.responses import ResponseTextDeltaEvent

async def demo_streaming():
    streamed = Runner.run_streamed(
        agente_simple,
        "Dame 3 consejos prácticos para viajar solo por primera vez a Asia."
    )
    async for evt in streamed.stream_events():
        if evt.type == "raw_response_event" and isinstance(evt.data, ResponseTextDeltaEvent):
            print(evt.data.delta, end="", flush=True)
    print()

asyncio.run(demo_streaming())

### 12.3 Context tipado

En producción necesitas pasar datos del usuario a las tools (su ID, sus preferencias guardadas, una conexión a base de datos...) sin que el modelo los vea directamente. El patrón es:

1. Defines un `dataclass` con tus datos.
2. Lo pasas a `Runner.run_sync(..., context=mis_datos)`.
3. Las tools lo reciben como primer parámetro (`RunContextWrapper`).

**Caso de uso típico**: personalizar la respuesta según el historial del usuario autenticado, sin meter todo el perfil en el prompt.

> **Ojo:** el contexto no se envía al modelo por sí solo, pero **lo que devuelve una tool sí llega al modelo**. Si la tool devuelve el historial completo, el modelo lo ve. Devuelve solo los campos que necesite.

In [ ]:
from dataclasses import dataclass
from agents import RunContextWrapper

@dataclass
class PerfilViajero:
    user_id: str
    nombre: str
    viajes_anteriores: list[str]

@function_tool
def historial_viajero(ctx: RunContextWrapper[PerfilViajero]) -> dict:
    """Devuelve el historial de viajes del usuario actual."""
    p = ctx.context
    return {"nombre": p.nombre, "viajes": p.viajes_anteriores, "total": len(p.viajes_anteriores)}

agente_personalizado = Agent[PerfilViajero](
    name="TravelMind Personalizado",
    instructions=(
        "Consulta el historial del viajero antes de recomendar. "
        "No repitas destinos que ya haya visitado. Personaliza por nombre."
    ),
    tools=[historial_viajero],
)

perfil = PerfilViajero(user_id="U-1042", nombre="María", viajes_anteriores=["París", "Roma", "Berlín", "Lisboa"])

resultado = Runner.run_sync(
    agente_personalizado,
    "¿Qué ciudad europea me recomiendas para el próximo puente?",
    context=perfil,
)
print(resultado.final_output)

### 12.4 Hosted tools y MCP

Además de tus propias funciones Python, el SDK da acceso a **tools alojadas en la infraestructura de OpenAI** — no necesitas implementarlas tú:

- `WebSearchTool()` — búsqueda web en tiempo real
- `FileSearchTool(...)` — RAG sobre documentos subidos a OpenAI
- `CodeInterpreterTool()` — ejecutar Python en un sandbox

Y con **MCP** (Model Context Protocol) puedes conectar el agente a cualquier servidor externo de tools — bases de datos, APIs internas, sistemas legacy — usando un estándar abierto. En el Bloque B usaremos esta idea para conectar el agente con las guías de viaje.

In [ ]:
from agents import WebSearchTool

agente_web = Agent(
    name="TravelMind Web",
    instructions=(
        "Usa búsqueda web para responder preguntas que requieran información actualizada. "
        "Cita siempre la fuente (dominio) al final de tu respuesta."
    ),
    tools=[WebSearchTool()],
)

resultado = Runner.run_sync(
    agente_web,
    "¿Necesitan visado los ciudadanos españoles para entrar a Japón en 2026?"
)
print(resultado.final_output)

## 13. Ejemplo integrador: TravelMind Assistant

Hasta aquí hemos visto cada pieza por separado. Ahora las juntamos todas en un sistema real.

**Lo que vamos a construir**: un asistente de viajes con arquitectura multi-agente que:

1. **Valida** la consulta antes de procesarla (guardrail de entrada)
2. **Consulta datos** con tools (vuelos y hoteles con precios mock)
3. **Delega** a especialistas según el tema (handoffs)
4. **Recuerda** el contexto entre turnos (session)

```
Viajero
   |
   v
[Input Guardrail] -- consulta fuera de viajes --> rechazada
   |
   v
[Coordinador TravelMind]
   |-- tool: buscar_vuelos()
   |-- tool: buscar_hoteles()
   |-- handoff --> [Especialista Visados]
   +-- handoff --> [Especialista Actividades]
   |
   v
[SQLiteSession] -- recuerda todo el hilo
```

Ejecuta las celdas una a una y observa en los prints qué está pasando en cada momento.

### Paso 1 — Tools: vuelos y hoteles

Dos funciones con datos mock que simulan consultas a una API real.
El agente las llamará automáticamente cuando la pregunta lo requiera — sin que se lo digamos explícitamente.

In [ ]:
from agents import Agent, Runner, function_tool, handoff, input_guardrail, GuardrailFunctionOutput, RunContextWrapper, SQLiteSession

# Datos mock — en producción serían llamadas a una API real
VUELOS = {
    ("Madrid", "Tokyo"):    {"precio": 780, "aerolinea": "Iberia/JAL",    "duracion_h": 14},
    ("Madrid", "Bangkok"):  {"precio": 520, "aerolinea": "Qatar Airways",  "duracion_h": 11},
    ("Madrid", "Lisboa"):   {"precio":  65, "aerolinea": "Iberia Express", "duracion_h":  1},
    ("Madrid", "Tokio"):    {"precio": 780, "aerolinea": "Iberia/JAL",    "duracion_h": 14},
}

HOTELES = {
    "Tokyo":   [{"nombre": "APA Hotel Shinjuku", "precio_noche":  85, "estrellas": 3},
                {"nombre": "Park Hyatt Tokyo",   "precio_noche": 420, "estrellas": 5}],
    "Bangkok": [{"nombre": "Lub d Silom",        "precio_noche":  28, "estrellas": 2},
                {"nombre": "Mandarin Oriental",  "precio_noche": 380, "estrellas": 5}],
    "Lisboa":  [{"nombre": "Yes! Lisbon Hostel", "precio_noche":  22, "estrellas": 1},
                {"nombre": "Bairro Alto Hotel",  "precio_noche": 310, "estrellas": 5}],
}

@function_tool
def buscar_vuelos(origen: str, destino: str) -> dict:
    """Busca vuelos disponibles entre dos ciudades y devuelve precio y aerolínea.

    Args:
        origen: Ciudad de origen del vuelo.
        destino: Ciudad de destino del vuelo.
    """
    print(f"  [tool] buscar_vuelos({origen} -> {destino})")
    clave = (origen.capitalize(), destino.capitalize())
    if clave in VUELOS:
        return VUELOS[clave]
    return {"error": f"No hay vuelos directos de {origen} a {destino} en nuestra base de datos."}

@function_tool
def buscar_hoteles(ciudad: str, max_precio_noche: float = 9999) -> list:
    """Busca hoteles en una ciudad filtrando por precio máximo por noche.

    Args:
        ciudad: Ciudad donde buscar alojamiento.
        max_precio_noche: Precio máximo por noche en euros (opcional).
    """
    print(f"  [tool] buscar_hoteles({ciudad}, max={max_precio_noche} EUR/noche)")
    ciudad_cap = ciudad.capitalize()
    hoteles = HOTELES.get(ciudad_cap, [])
    filtrados = [h for h in hoteles if h["precio_noche"] <= max_precio_noche]
    if not filtrados:
        return [{"mensaje": f"No hay hoteles en {ciudad} dentro de ese presupuesto."}]
    return filtrados

print("Tools definidas: buscar_vuelos, buscar_hoteles")

### Paso 2 — Especialistas con handoff

Dos agentes especializados. El coordinador les cederá el control cuando detecte
que la pregunta encaja con su dominio — sin lógica de enrutamiento manual.

In [ ]:
especialista_visados = Agent(
    name="Especialista en Visados",
    instructions=(
        "Eres experto en visados y documentacion de viaje para ciudadanos espanoles. "
        "Explica con precision los requisitos de entrada, duracion de estancia permitida "
        "y cualquier tramite previo necesario. Si no tienes datos actualizados, recomienda "
        "verificar en la embajada correspondiente."
    )
)

especialista_actividades = Agent(
    name="Especialista en Actividades",
    instructions=(
        "Eres experto en planes, excursiones y experiencias de viaje. "
        "Recomienda actividades concretas adaptadas al perfil del viajero (familia, mochilero, lujo...). "
        "Incluye siempre una estimacion de precio y la duracion aproximada de cada actividad."
    )
)

print("Especialistas definidos: visados y actividades")

### Paso 3 — Guardrail: solo consultas de viajes

Antes de que cualquier mensaje llegue al coordinador, el guardrail clasifica la consulta con un **agente LLM dedicado** (pequeño y barato). Si no tiene que ver con viajes, la corta antes de que el coordinador empiece, porque lo configuramos como bloqueante. El clasificador sí gasta unos pocos tokens: es un LLM pequeño.

> Un guardrail de keywords falla con preguntas como "¿cuál es la capital de Francia?" — no contiene ninguna keyword prohibida pero claramente es off-topic. Un clasificador LLM entiende la intención, no las palabras exactas.

In [ ]:
from pydantic import BaseModel as _BM

class _TopicCheck(_BM):
    es_viajes: bool
    motivo: str

_clasificador_topic = Agent(
    name="Clasificador de tópico",
    instructions=(
        "Responde si la consulta del usuario está relacionada con viajes, turismo, "
        "destinos, transporte, alojamiento o planificación de viajes. "
        "Devuelve es_viajes=true aunque la pregunta sea vaga o use sinónimos. "
        "Devuelve es_viajes=false si la pregunta no tiene nada que ver con viajes."
    ),
    model="gpt-6-luna",
    output_type=_TopicCheck,
)

@input_guardrail(run_in_parallel=False)   # bloqueante: el coordinador no empieza hasta que el filtro da el visto bueno
async def solo_viajes(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input_text,
) -> GuardrailFunctionOutput:
    """Rechaza consultas que no tengan relación con viajes o turismo."""
    def extraer_texto(obj) -> str:
        if isinstance(obj, str):
            return obj
        if isinstance(obj, list):
            return " ".join(extraer_texto(item) for item in obj)
        if isinstance(obj, dict):
            return extraer_texto(obj.get("content", "") or obj.get("text", ""))
        return str(obj)

    texto = extraer_texto(input_text)
    res = await Runner.run(_clasificador_topic, texto)
    check = res.final_output
    return GuardrailFunctionOutput(output_info=check.motivo, tripwire_triggered=not check.es_viajes)

print("Guardrail definido: solo_viajes (clasificador LLM)")

### Paso 4 — Coordinador: el agente central

El coordinador recibe todas las consultas. Usa las tools para datos concretos
y hace handoff a los especialistas cuando el tema lo requiere.
El guardrail se aplica aquí, antes de que la consulta llegue al modelo.

In [ ]:
coordinador = Agent(
    name="TravelMind Coordinador",
    instructions=(
        "Eres TravelMind, el asistente central de una agencia de viajes inteligente. "
        "Tu trabajo es dar respuestas completas y practicas. Tienes estas capacidades:\n"
        "- Usa buscar_vuelos() cuando te pregunten por precios o disponibilidad de vuelos.\n"
        "- Usa buscar_hoteles() cuando te pregunten por alojamiento.\n"
        "- Delega al Especialista en Visados para requisitos de entrada y documentacion.\n"
        "- Delega al Especialista en Actividades para planes, excursiones o que hacer.\n"
        "Se concreto y usa los datos que devuelven las tools — no inventes precios."
    ),
    tools=[buscar_vuelos, buscar_hoteles],
    handoffs=[
        handoff(especialista_visados),
        handoff(especialista_actividades),
    ],
    input_guardrails=[solo_viajes],
)

print("Coordinador listo: tools + handoffs + guardrail")

### Paso 5 — Primera consulta: vuelo + hotel a la vez

La pregunta necesita dos tools. Observa en los prints que el agente las llama
en paralelo — no espera a una para lanzar la otra.

In [ ]:
from agents.exceptions import InputGuardrailTripwireTriggered

sesion = SQLiteSession("travelmind_demo_clase")

def preguntar(consulta: str):
    """Envía una consulta al coordinador dentro de la sesión activa."""
    print(f"\nUsuario: {consulta}")
    print("-" * 55)
    try:
        resultado = Runner.run_sync(coordinador, consulta, session=sesion)
        print(resultado.final_output)
    except InputGuardrailTripwireTriggered as e:
        print(f"[Bloqueado por el guardrail] {e.guardrail_result.output.output_info}")

# Necesita buscar_vuelos Y buscar_hoteles — fíjate en cuántas tools llama
preguntar("¿Cuánto cuesta el vuelo Madrid-Tokio y qué hoteles hay por menos de 150 EUR la noche?")

### Paso 6 — El coordinador delega a un especialista

Ahora preguntamos algo fuera del dominio del coordinador.
Detecta que encaja con visados y hace un handoff — el Especialista en Visados responde.

In [ ]:
# El coordinador no sabe de visados en detalle -> handoff al especialista
preguntar("¿Necesito visado para entrar en Japón con pasaporte español?")

### Paso 7 — La sesión recuerda el contexto

Preguntamos algo vago que solo tiene sentido si se recuerdan los turnos anteriores.
Gracias a SQLiteSession el agente sabe que estamos hablando de Japón y Tokio.

In [ ]:
# Sin sesión esta pregunta sería ambigua; con sesión el agente la entiende
preguntar("¿Y qué actividades me recomiendas para esa ciudad?")

### Paso 8 — El guardrail en acción

Enviamos una consulta que no tiene nada que ver con viajes.
El guardrail la para antes de que el coordinador empiece a trabajar, porque lo hemos configurado con `run_in_parallel=False`. No es gratis: el filtro es un LLM pequeño, así que gasta unos pocos tokens y tarda un poco.

> Por defecto, los guardrails de entrada corren **en paralelo** con el agente: la respuesta llega antes, pero el agente puede haber empezado (y gastado tokens, o llamado a una tool) cuando el guardrail salta. Si el filtro debe impedir que algo se ejecute, usa `run_in_parallel=False`.

In [ ]:
# Fuera del ambito -> el guardrail lo bloquea sin llamar al modelo
preguntar("Cual es la capital de Francia y como se calcula el area de un triangulo?")

### Qué hemos visto en este ejemplo

| Pieza | Dónde aparece |
|---|---|
| **Agent** | Coordinador + 2 especialistas |
| **function_tool** | `buscar_vuelos`, `buscar_hoteles` |
| **handoff** | Coordinador delega a Visados / Actividades |
| **input_guardrail** | `solo_viajes` filtra consultas ajenas |
| **SQLiteSession** | Recuerda el hilo entre los 4 turnos |

Este es exactamente el patrón que usarás en el **Reto del Asistente de Guías de Viaje** —
pero con RAG real sobre PDFs en lugar de datos mock.

## Resumen

| Concepto | En una frase | Cuándo usarlo |
|---|---|---|
| **Agent** | LLM con instrucciones y herramientas | Siempre — es la base |
| **Tools** | Funciones Python que el agente puede invocar | Cuando necesitas que el agente *haga* algo |
| **Handoffs** | Un agente pasa el control a otro más especializado | Sistemas multi-agente |
| **Guardrails** | Filtros de entrada y salida | Control de calidad y seguridad |
| **Sessions** | Historial conversacional automático | Diálogos multi-turno |
| **output_type** | La respuesta es un objeto tipado, no texto libre | Cuando procesas la respuesta en código |

## Ejemplos extra para explorar

Los tres bloques siguientes son ejemplos ya implementados que puedes ejecutar y modificar libremente.
Úsalos para experimentar antes del Reto.

### Ejemplo A — Conversor de unidades de viaje

Tres tools independientes. El agente recibe una sola pregunta con 3 conversiones distintas
y decide llamarlas todas — sin que le digamos explícitamente cuáles usar.

Fíjate en los prints: verás que las 3 tools se ejecutan antes de que aparezca la respuesta.

In [ ]:
from datetime import date
from agents import Agent, Runner, function_tool

@function_tool
def celsius_a_fahrenheit(temp_c: float) -> float:
    """Convierte temperatura de Celsius a Fahrenheit.
    Args:
        temp_c: Temperatura en grados Celsius.
    """
    print(f"  -> celsius_a_fahrenheit({temp_c})")
    return round(temp_c * 9 / 5 + 32, 1)

@function_tool
def kg_a_libras(kg: float) -> float:
    """Convierte el peso de kilos a libras (útil para límites de equipaje).
    Args:
        kg: Peso en kilogramos.
    """
    print(f"  -> kg_a_libras({kg})")
    return round(kg * 2.20462, 2)

@function_tool
def dias_hasta(fecha_salida: str) -> int:
    """Calcula cuántos días quedan hasta la fecha de salida.
    Args:
        fecha_salida: Fecha en formato YYYY-MM-DD.
    """
    print(f"  -> dias_hasta({fecha_salida})")
    return (date.fromisoformat(fecha_salida) - date.today()).days

agente_conversor = Agent(
    name="TravelMind Conversor",
    instructions="Eres un asistente de conversiones para viajeros. Usa las tools para dar datos exactos.",
    tools=[celsius_a_fahrenheit, kg_a_libras, dias_hasta],
)

resultado = Runner.run_sync(
    agente_conversor,
    "Bangkok está a 34 °C, mi maleta pesa 23 kg y me voy el 2026-08-10. ¿En Fahrenheit, libras y cuántos días?"
)
print(resultado.final_output)

### Ejemplo B — Sistema multi-especialista con perfil de usuario

Dos niveles de lógica en una sola pregunta:

1. El coordinador llama a `consultar_perfil_viajero` para saber quién es el usuario
2. Con ese contexto, hace **handoff** al Especialista en Alojamiento

Prueba a cambiar `U-001` por `U-002` y observa cómo cambia la recomendación
para Laura (preferencia "lujo") vs Carlos (preferencia "económico").

In [ ]:
from agents import Agent, Runner, function_tool, handoff

PERFILES = {
    "U-001": {"nombre": "Carlos", "preferencia": "economico", "viajes": ["Paris", "Berlin"]},
    "U-002": {"nombre": "Laura",  "preferencia": "lujo",      "viajes": ["Dubai", "Maldivas"]},
}

@function_tool
def consultar_perfil_viajero(user_id: str) -> dict:
    """Devuelve el perfil del viajero: nombre, preferencia de viaje y destinos anteriores.
    Args:
        user_id: Identificador del usuario (ej: 'U-001').
    """
    print(f"  -> consultar_perfil_viajero({user_id})")
    return PERFILES.get(user_id, {"nombre": "Desconocido", "preferencia": "medio", "viajes": []})

esp_vuelos = Agent(
    name="Especialista en Vuelos",
    instructions="Eres experto en vuelos, aerolíneas y precios. Adapta la recomendación al perfil del viajero.",
)
esp_hotel = Agent(
    name="Especialista en Alojamiento",
    instructions="Eres experto en hoteles y alojamiento. Adapta según el presupuesto del viajero.",
)
esp_activ = Agent(
    name="Especialista en Actividades",
    instructions="Eres experto en planes y experiencias de viaje. Personaliza según los intereses del viajero.",
)

coordinador_perfil = Agent(
    name="TravelMind Coordinador con Perfil",
    instructions=(
        "Antes de responder, consulta siempre el perfil del viajero con consultar_perfil_viajero. "
        "Luego delega al especialista apropiado según el tema de la consulta."
    ),
    tools=[consultar_perfil_viajero],
    handoffs=[handoff(esp_vuelos), handoff(esp_hotel), handoff(esp_activ)],
)

# Cambia el user_id a 'U-002' para ver cómo cambia la respuesta para Laura
resultado = Runner.run_sync(
    coordinador_perfil,
    "Soy el usuario U-001. ¿Qué hotel me recomiendas en Lisboa?"
)
print(resultado.final_output)

### Ejemplo C — Agente personalizado con context tipado y session

Las tres piezas avanzadas juntas:

- **`context` tipado**: el perfil de Ana no va en el prompt; el agente lo pide con una tool (y lo que la tool devuelve sí lo ve el modelo)
- **`registrar_destino`**: modifica el estado del contexto en tiempo real (lista de visitados)
- **`SQLiteSession`**: el historial persiste entre los 3 turnos

Lo clave del turno 3: la pregunta "¿el primero que dijiste?" es ambigua sin memoria.
El agente sabe que fue Lisboa porque lo tiene en el historial de la sesión.

In [ ]:
from dataclasses import dataclass, field
from agents import Agent, Runner, function_tool, SQLiteSession, RunContextWrapper

@dataclass
class PerfilViajero:
    nombre: str
    presupuesto_total: float
    destinos_visitados: list[str] = field(default_factory=list)
    preferencias: list[str] = field(default_factory=list)

@function_tool
def obtener_perfil(ctx: RunContextWrapper[PerfilViajero]) -> dict:
    """Devuelve el perfil completo del viajero actual."""
    p = ctx.context
    print(f"  -> obtener_perfil() — {p.nombre}, visitados: {p.destinos_visitados}")
    return {
        "nombre": p.nombre,
        "presupuesto": p.presupuesto_total,
        "preferencias": p.preferencias,
        "ya_visitados": p.destinos_visitados,
    }

@function_tool
def registrar_destino(ctx: RunContextWrapper[PerfilViajero], destino: str) -> str:
    """Anota el destino recomendado para no volver a sugerirlo.
    Args:
        destino: Nombre del destino a registrar.
    """
    ctx.context.destinos_visitados.append(destino)
    print(f"  -> registrar_destino({destino}) — total: {ctx.context.destinos_visitados}")
    return f"{destino} registrado. Ya visitados: {ctx.context.destinos_visitados}"

perfil = PerfilViajero(
    nombre="Ana",
    presupuesto_total=2000,
    preferencias=["cultura", "gastronomia"],
)
sesion = SQLiteSession("ana_planificacion_demo_2")

agente_personal = Agent[PerfilViajero](
    name="TravelMind Personal",
    instructions=(
        "Consulta siempre el perfil del viajero con obtener_perfil antes de responder. "
        "Personaliza la respuesta usando su nombre. "
        "No repitas destinos que aparezcan en ya_visitados. "
        "Cuando recomiendes un destino concreto, regístralo con registrar_destino."
    ),
    tools=[obtener_perfil, registrar_destino],
)

# Turno 1 — recomienda un destino y lo registra
r1 = Runner.run_sync(agente_personal, "¿Qué destino europeo me recomiendas para este verano?", session=sesion, context=perfil)
print("Turno 1:", r1.final_output)

# Turno 2 — no puede repetir el destino del turno 1
r2 = Runner.run_sync(agente_personal, "¿Y para otoño? Dame otra opción diferente.", session=sesion, context=perfil)
print("\nTurno 2:", r2.final_output)

# Turno 3 — pregunta vaga que requiere memoria de los turnos anteriores
r3 = Runner.run_sync(agente_personal, "¿Qué actividades haría por las tardes en el primero que dijiste?", session=sesion, context=perfil)
print("\nTurno 3:", r3.final_output)

### Ejemplo D — MCP: conectar a un servidor externo de tools

Hasta ahora todas las tools eran funciones Python definidas en el propio notebook.
**MCP** (Model Context Protocol) permite que el agente descubra y use tools que viven
en un servidor externo — sin que tú escribas ninguna función.

El flujo es el mismo que explica la diapositiva:

```
Agente  --tools/list-->  Servidor MCP  (descubre qué tools hay)
Agente  --tools/call-->  Servidor MCP  (ejecuta una tool concreta)
        <-- resultado --
```

Usamos el servidor público **DeepWiki** (`mcp.deepwiki.com/mcp`), que responde
preguntas sobre cualquier repositorio de GitHub. El agente descubrira las tools
disponibles solo al arrancar — no hay nada que configurar.

In [ ]:
import asyncio
from agents import Agent, Runner
from agents.mcp import MCPServerStreamableHttp

# Servidor MCP público: ya está en marcha, no hay que levantar nada
MCP_URL = "https://mcp.deepwiki.com/mcp"

async def demo_mcp():
    async with MCPServerStreamableHttp(params={"url": MCP_URL}, name="DeepWiki") as servidor_mcp:

        # El agente descubre automáticamente las tools del servidor al iniciarse
        agente_mcp = Agent(
            name="TravelMind MCP",
            instructions=(
                "Usas las tools disponibles para responder preguntas sobre repositorios de GitHub. "
                "Cita siempre el repositorio que has consultado."
            ),
            mcp_servers=[servidor_mcp],
        )

        print("Tools disponibles en el servidor MCP:")
        tools = await servidor_mcp.list_tools()
        for t in tools:
            print(f"  - {t.name}: {t.description[:60]}...")

        print()
        resultado = await Runner.run(
            agente_mcp,
            "¿Qué hace el repositorio openai/openai-agents-python? Dame un resumen en 3 frases."
        )
        print("Respuesta:", resultado.final_output)

asyncio.run(demo_mcp())

## Recursos

- **OpenAI Agents SDK**: https://openai.github.io/openai-agents-python/
- **Repo GitHub**: https://github.com/openai/openai-agents-python
- **Pydantic AI** (alternativa): https://ai.pydantic.dev
- **Anthropic — Building effective agents**: https://www.anthropic.com/research/building-effective-agents